In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, recall_score
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

# 1. 데이터 불러오기
try:
    pest_df = pd.read_csv("통합_논벼_예찰결과_피해율유지.csv", encoding='utf-8-sig')
    weather_df = pd.read_csv("전남_일별_기상청_데이터_최종수정.csv", encoding='utf-8-sig')
except:
    pest_df = pd.read_csv("통합_논벼_예찰결과_피해율유지.csv", encoding='cp949')
    weather_df = pd.read_csv("전남_일별_기상청_데이터_최종수정.csv", encoding='cp949')

pest_df['조사일'] = pd.to_datetime(pest_df['조사일'].astype(str).str.replace('.', '-'))
weather_df['일시'] = pd.to_datetime(weather_df['일시'])

# 🌟 6월 제외 (7, 8, 9월 데이터만 사용)
pest_df = pest_df[pest_df['조사일'].dt.month.isin([7, 8, 9])]

pest_df['매칭지역'] = pest_df['지역'].astype(str).str.replace(r'[시군]$', '', regex=True)
weather_df['매칭지역'] = weather_df['지점명'].astype(str)

# ==========================================================
# 🌟 기상 데이터 피처 생성 (결측치 원천 차단)
# ==========================================================
weather_cols = ['평균기온(°C)', '최고기온(°C)', '최저기온(°C)', '평균 상대습도(%)', 
                '일강수량(mm)', '합계 일조시간(hr)', '평균 풍속(m/s)', '평균 이슬점온도(°C)']

valid_cols = [c for c in weather_cols if c in weather_df.columns]

# 💡 핵심 수정: 결측치가 하나라도 있는 날짜의 기상 데이터는 아예 빼버림 (왜곡 원천 차단)
weather_df = weather_df.dropna(subset=valid_cols)
weather_df = weather_df.sort_values(['매칭지역', '일시']).reset_index(drop=True)

wg = weather_df.groupby('매칭지역')

# 살아남은 깨끗한 데이터들로만 14일 치 롤링 계산
weather_df['평균기온_mean']   = wg['평균기온(°C)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['최고기온_mean']   = wg['최고기온(°C)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['최저기온_mean']   = wg['최저기온(°C)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['습도_mean']       = wg['평균 상대습도(%)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['강수량_mean']     = wg['일강수량(mm)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['일조시간_mean']   = wg['합계 일조시간(hr)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['평균풍속_mean']   = wg['평균 풍속(m/s)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['이슬점온도_mean'] = wg['평균 이슬점온도(°C)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)

weather_df['최고기온_max']    = wg['최고기온(°C)'].rolling(14, min_periods=1).max().reset_index(level=0, drop=True)
weather_df['최저기온_min']    = wg['최저기온(°C)'].rolling(14, min_periods=1).min().reset_index(level=0, drop=True)
weather_df['강수량_sum']      = wg['일강수량(mm)'].rolling(14, min_periods=1).sum().reset_index(level=0, drop=True)
weather_df['일조시간_sum']    = wg['합계 일조시간(hr)'].rolling(14, min_periods=1).sum().reset_index(level=0, drop=True)

def get_max_consecutive_rain(series):
    is_rainy = series > 0
    if not is_rainy.any(): return 0
    return is_rainy.groupby((~is_rainy).cumsum()).sum().max()

weather_df['연속강수일수'] = wg['일강수량(mm)'].rolling(14, min_periods=1).apply(get_max_consecutive_rain, raw=False).reset_index(level=0, drop=True)

feature_list = ['평균기온_mean', '최고기온_mean', '최저기온_mean', '습도_mean', 
                '강수량_mean', '일조시간_mean', '평균풍속_mean', '이슬점온도_mean',
                '최고기온_max', '최저기온_min', '강수량_sum', '일조시간_sum', '연속강수일수']

w_features = weather_df[['매칭지역', '일시'] + feature_list]

# ==========================================================
# 🌟 모든 병해충 모델 스캔 및 엄격한 데이터 필터링
# ==========================================================
target_diseases = [col for col in pest_df.columns if '(피해율)' in col]
all_results = []

print("⏳ 결측치 제거 후 순수 데이터 모델 스캔 중... (엄격한 테스트 검증 적용)\n")

for target in target_diseases:
    df_subset = pest_df[['조사일', '매칭지역', target]].dropna()
    if len(df_subset) == 0: continue
        
    positive_damage = df_subset[df_subset[target] > 0][target]
    if len(positive_damage) == 0: continue 
    
    median_val = positive_damage.median()
    
    def grade_by_median(rate):
        if rate == 0: return 0
        elif rate <= median_val: return 1
        else: return 2
        
    df_subset['damage_grade'] = df_subset[target].apply(grade_by_median)
    merged_df = pd.merge(df_subset, w_features, left_on=['조사일', '매칭지역'], right_on=['일시', '매칭지역'], how='inner').dropna()
    
    # 평가 조건 컷오프 (0, 1, 2 등급이 모두 존재 & 가장 적은 등급이 최소 5개 이상 보장)
    class_counts = merged_df['damage_grade'].value_counts()
    if len(class_counts) < 3 or class_counts.min() < 5: 
        continue
        
    X = merged_df[feature_list]
    y = merged_df['damage_grade']
    
    # 층화 추출 강제. 실패 시 무작위 분할 대신 과감히 패스
    try:
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    except ValueError:
        continue
        
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # SMOTE 적용
    min_class_count = y_train.value_counts().min()
    k_neighbors = min(5, min_class_count - 1) if min_class_count > 1 else 1
    
    if min_class_count > 1:
        smote = SMOTE(k_neighbors=k_neighbors, random_state=42)
        X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)
    else:
        X_train_resampled, y_train_resampled = X_train_scaled, y_train
        
    models = {
        "LR": LogisticRegression(max_iter=1000, random_state=42),
        "KNN": KNeighborsClassifier(n_neighbors=5),
        "RF": RandomForestClassifier(n_estimators=100, max_depth=7, random_state=42),
        "XGB": XGBClassifier(random_state=42, eval_metric='mlogloss')
    }
    
    clean_name = target.replace(' (피해율)', '')
    
    for model_name, model in models.items():
        model.fit(X_train_resampled, y_train_resampled)
        y_pred = model.predict(X_test_scaled)
        
        acc = accuracy_score(y_test, y_pred) * 100
        recalls = recall_score(y_test, y_pred, labels=[0, 1, 2], average=None, zero_division=0) * 100
        
        all_results.append({
            '병해충명': clean_name,
            '총 데이터수': len(merged_df),
            '중앙값(구분값)': round(median_val, 2),
            '모델': model_name,
            '정확도': round(acc, 1),
            '정상(0) 재현율': round(recalls[0], 1),
            '경미(1) 재현율': round(recalls[1], 1),
            '심각(2) 재현율': round(recalls[2], 1)
        })

# ==========================================================
# 🌟 결과 출력
# ==========================================================
results_df = pd.DataFrame(all_results)

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("🔥 검증 완료: 순수 기상 데이터(결측치 제거) 기반 '진짜' 모델 성능 🔥")
print("-" * 105)
if len(results_df) > 0:
    results_df = results_df.sort_values(by=['병해충명', '심각(2) 재현율'], ascending=[True, False])
    print(results_df.to_string(index=False))
else:
    print("⚠️ 분석 가능한 요건(결측치 제외 후 각 등급 데이터 최소 5개 이상)을 충족하는 병해충 데이터가 없습니다.")
print("-" * 105)

⏳ 결측치 제거 후 순수 데이터 모델 스캔 중... (엄격한 테스트 검증 적용)

🔥 검증 완료: 순수 기상 데이터(결측치 제거) 기반 '진짜' 모델 성능 🔥
---------------------------------------------------------------------------------------------------------
⚠️ 분석 가능한 요건(결측치 제외 후 각 등급 데이터 최소 5개 이상)을 충족하는 병해충 데이터가 없습니다.
---------------------------------------------------------------------------------------------------------


### [데이터 전처리]

In [10]:
import pandas as pd

# 1. 데이터 로드
df = pd.read_csv('통합_논벼_예찰결과_피해율유지.csv')

# 2. 롱 포맷으로 변환 (melt 사용)
# id_vars: 유지할 컬럼 / value_vars: 병해충 이름으로 녹여낼 컬럼들
id_columns = ['조사일', '지역']
pest_columns = df.columns.difference(id_columns)

melted_df = df.melt(id_vars=id_columns, 
                    value_vars=pest_columns, 
                    var_name='병해충', 
                    value_name='피해율')

# 3. 발생여부 컬럼 생성
# 피해율이 0보다 크면 1, 아니면 0 (NaN은 0으로 처리하거나 제외 가능)
melted_df['발생여부'] = melted_df['피해율'].apply(lambda x: 1 if x > 0 else 0)

# 4. 최종 컬럼 정리 (요청하신 4개 컬럼만 추출)
final_df = melted_df[['조사일', '지역', '병해충', '발생여부']]

# 5. 결과 확인
print(f"변환 전 데이터 크기: {df.shape}")
print(f"변환 후 데이터 크기: {final_df.shape}")
print(final_df.head())

# 필요한 경우 파일로 저장
final_df.to_csv('전처리_논벼_병해충_발생여부.csv', index=False, encoding='utf-8-sig')

변환 전 데이터 크기: (1758, 25)
변환 후 데이터 크기: (40434, 4)
          조사일   지역    병해충  발생여부
0  2014.09.16  강진군  깨씨무늬병     0
1  2014.09.16  고흥군  깨씨무늬병     1
2  2014.09.16  곡성군  깨씨무늬병     1
3  2014.09.16  광양시  깨씨무늬병     1
4  2014.09.16  구례군  깨씨무늬병     0


In [8]:
gangjin_df = final_df[final_df['지역'] == '강진군']

gangjin_df

,조사일,지역,병해충,발생여부
0,2014.09.16,강진군,깨씨무늬병,0
21,2015.06.16,강진군,깨씨무늬병,0
42,2015.07.01,강진군,깨씨무늬병,0
63,2015.07.16,강진군,깨씨무늬병,0
84,2015.08.01,강진군,깨씨무늬병,0
105,2015.08.16,강진군,깨씨무늬병,1
126,2016.06.01,강진군,깨씨무늬병,0
147,2016.06.16,강진군,깨씨무늬병,0
168,2016.07.01,강진군,깨씨무늬병,0
189,2016.07.16,강진군,깨씨무늬병,0
